In [ ]:
from data_engineering.flights.flight_type import get_flights
from datetime import date
import polars as pl

from data_engineering.flights.sfdps_to_flights import get_sfdps_flights_day
df = get_sfdps_flights_day(date(2026,3,1))

In [12]:
from datetime import date
from flights.evaluation.evaluation import main
results, stats = main(
    test_src="algorithm-no-airports",
    gold_src="ground_truth",
    test_dates=date(2026, 3, 1),
    adsb_src="adsblol",  # Internal name for adsb.lol
)

Stats for algorithm-no-airports:ground_truth: {'true_positive': 2247, 'false_positive': 1, 'false_negative': 2, 'precision': 0.9995551601423488, 'recall': 0.9991107158737217, 'f1': 0.9993328885923949, 'takeoff_airport_ident_total_count': 2247, 'takeoff_airport_ident_match_count': 2238, 'takeoff_airport_ident_incorrect_match_count': 9, 'takeoff_airport_ident_match_pct': 0.9959946595460614, 'landing_airport_ident_total_count': 2247, 'landing_airport_ident_match_count': 2222, 'landing_airport_ident_incorrect_match_count': 25, 'landing_airport_ident_match_pct': 0.9888740542946151, 'airport_ident_total_count': 2247, 'airport_ident_match_count': 2213, 'airport_ident_incorrect_match_count': 34, 'airport_ident_match_pct': 0.9848687138406764}
Stats CSV saved to /Volumes/T2-SSD/planequery/data/experiments/comparison/2026-09-22_01-01_cf2ca18_flights_evaluation.csv


In [13]:
df = results[0]

In [14]:
df.columns

['icao',
 'callsign',
 'registration',
 'takeoff_time',
 'takeoff_airport_ident',
 'landing_time',
 'landing_airport_ident',
 'first_message_time',
 'first_lat',
 'first_lon',
 'first_baro_altitude_ft',
 'first_geom_altitude_ft',
 'last_message_time',
 'last_lat',
 'last_lon',
 'last_baro_altitude_ft',
 'last_geom_altitude_ft',
 'pia',
 'ladd',
 'military',
 'interesting',
 'aircraft_type',
 'owner',
 'aircraft_description',
 'category',
 'icao_df1',
 'callsign_df1',
 'registration_df1',
 'takeoff_time_df1',
 'takeoff_airport_ident_df1',
 'landing_time_df1',
 'landing_airport_ident_df1',
 'pia_df1',
 'ladd_df1',
 'military_df1',
 'interesting_df1',
 'aircraft_type_df1',
 'owner_df1',
 'aircraft_description_df1',
 'category_df1',
 'takeoff_airport_in_us_faa_airspace',
 'landing_airport_in_us_faa_airspace',
 'takeoff_airport_in_us_faa_island_airspace',
 'landing_airport_in_us_faa_island_airspace',
 'first_message_time_df1',
 'first_lat_df1',
 'first_lon_df1',
 'last_message_time_df1',
 '

In [7]:
import polars as pl
df = df.with_columns((pl.col("category") == pl.col("category_df1")).alias("category_match"))
df.filter(pl.col("match_status") == "both").get_column("category_match").value_counts()

category_match,count
bool,u32
false,1708
true,539


In [1]:
from flights.ground_truth.bts_and_sfdps_to_gold import get_ground_truth_flights_for_day
from datetime import date
from data_engineering.flights.flight_type import get_flights
df = get_flights(date(2026, 3, 1))

Loading airports from /Volumes/T2-SSD/planequery/data/raw/ourairports/airports.csv


In [5]:
from airports.polars_exprs import airport_has_iata_code_expr, airport_has_icao_code_expr, airport_size_expr
df = df.with_columns(airport_size_expr("takeoff_airport_ident")).with_columns(airport_has_icao_code_expr("takeoff_airport_ident")).with_columns(airport_size_expr("landing_airport_ident")).with_columns(airport_has_iata_code_expr("takeoff_airport_ident"))
df.columns

['icao',
 'callsign',
 'registration',
 'takeoff_time',
 'takeoff_airport_ident',
 'landing_time',
 'landing_airport_ident',
 'first_message_time',
 'first_lat',
 'first_lon',
 'first_baro_altitude_ft',
 'first_geom_altitude_ft',
 'last_message_time',
 'last_lat',
 'last_lon',
 'last_baro_altitude_ft',
 'last_geom_altitude_ft',
 'pia',
 'ladd',
 'military',
 'interesting',
 'aircraft_type',
 'owner',
 'aircraft_description',
 'category',
 'takeoff_airport_size',
 'takeoff_airport_has_icao_code',
 'landing_airport_size',
 'landing_airport_has_iata_code',
 'takeoff_airport_has_iata_code']

In [9]:
df.get_column("takeoff_airport_has_iata_code").value_counts()

takeoff_airport_has_iata_code,count
bool,u32
true,104508
false,23993


In [ ]:
df.get_column("takeoff_airport_has_icao_code").value_counts()

takeoff_airport_has_icao_code,count
bool,u32
true,106748
false,21753


In [ ]:
df = get_flights(date(2026,3,1))
df.filter(pl.col("category") == "B3")

In [ ]:
from data_engineering.eurocontrol.read import get_eurocontrol_flights
from datetime import date

df = get_eurocontrol_flights(date(2024,6,1))

In [ ]:
df

In [ ]:
from data_engineering.flights.flight_type import flight_airport_exprs
df.with_columns(flight_airport_exprs()).get_column("takeoff_airport_size").value_counts()

In [ ]:
df.with_columns(flight_airport_exprs())

In [ ]:
df.with_columns(flight_airport_exprs()).get_column("is_commerical_category").value_counts()

In [ ]:
cessnas = df.filter(
    pl.col("aircraft_description")
    .str.contains(r"(?i)\bcessna\b")
    .fill_null(False)
)
cessnas

In [ ]:
import polars as pl
CESSNA_TYPES = [
    # Single-engine piston
    "C150",
    "C152",
    "C162",
    "C170",
    "C172",
    "C175",
    "C177",
    "C180",
    "C182",
    "C185",
    "C188",
    "C190",
    "C195",
    "C205",
    "C206",
    "C207",
    "C210",

    # Twins / utility
    "C303",
    "C305",
    "C310",
    "C320",
    "C335",
    "C336",
    "C337",
    "C340",
    "C401",
    "C402",
    "C404",
    "C406",
    "C408",  # SkyCourier
    "C411",
    "C414",
    "C421",
    "C425",
    "C441",

    # Caravan
    "C208",

    # Citation jets
    "C500",  # Citation I
    "C501",  # Citation I/SP
    "C510",  # Citation Mustang
    "C525",  # CitationJet / CJ1
    "C25A",  # CJ2
    "C25B",  # CJ3
    "C25C",  # CJ4
    "C526",
    "C550",  # Citation II
    "C551",
    "C55B",  # Citation Bravo
    "C560",  # Citation V / Ultra / Encore
    "C56X",  # Citation Excel / XLS
    "C650",  # Citation III / VI / VII
    "C680",  # Citation Sovereign
    "C68A",  # Citation Latitude
    "C700",  # Citation Longitude
    "C750",  # Citation X
]
with pl.Config(tbl_rows=-1):
    display(df.with_columns(flight_airport_exprs()).filter(~pl.col("aircraft_type").is_in(CESSNA_TYPES)))

In [ ]:
import polars as pl
    # Single-engine piston
    "C150",
    "C152",
    "C162",
    "C170",
    "C172",
    "C175",
    "C177",
    "C180",
    "C182",
    "C185",
    "C188",
    "C190",
    "C195",
    "C205",
    "C206",
    "C207",
    "C210",

    # Twins / utility
    "C303",
    "C305",
    "C310",
    "C320",
    "C335",
    "C336",
    "C337",
    "C340",
    "C401",
    "C402",
    "C404",
    "C406",
    "C408",  # SkyCourier
    "C411",
    "C414",
    "C421",
    "C425",
    "C441",

    # Caravan
    "C208",

    # Citation jets
    "C500",  # Citation I
    "C501",  # Citation I/SP
    "C510",  # Citation Mustang
    "C525",  # CitationJet / CJ1
    "C25A",  # CJ2
    "C25B",  # CJ3
    "C25C",  # CJ4
    "C526",
    "C550",  # Citation II
    "C551",
    "C55B",  # Citation Bravo
    "C560",  # Citation V / Ultra / Encore
    "C56X",  # Citation Excel / XLS
    "C650",  # Citation III / VI / VII
    "C680",  # Citation Sovereign
    "C68A",  # Citation Latitude
    "C700",  # Citation Longitude
    "C750",  # Citation X
]
with pl.Config(tbl_rows=-1):
    display(df.with_columns(flight_airport_exprs()).filter(pl.col("is_commerical_category") == False))

In [ ]:
with pl.Config(tbl_rows=-1):
    display(df.get_column("category").value_counts())

In [ ]:
from data_engineering.flights.flight_type import is_commercial_category
df.filter((is_commercial_category()))